In [ ]:
import os
import pandas as pd
from glob import glob

from util import calc_perceptual_metrics


attack_dir = 'exp/attack'
percept_dir = 'exp/percept'

In [ ]:
res = []
for run_dir in os.listdir(attack_dir):
    lalm_name = run_dir.split('-')[0]
    attack_name = run_dir.split('-')[1]
    for job_name in os.listdir(os.path.join(attack_dir, run_dir)):
        wav_dir = os.path.join(attack_dir, run_dir, job_name, 'wav')
        adv_files = glob(os.path.join(wav_dir, 'adv_*.wav'))
        if len(adv_files) == 0:
            continue
        ben_file = glob(os.path.join(wav_dir, 'ben_*.wav'))[0]
        carrier_type = job_name.split('_')[3]
        for adv_file in adv_files:
            snr, mcd, stoi, pesq = calc_perceptual_metrics(ben_file, adv_file)
            res.append([
                lalm_name,
                attack_name, 
                carrier_type,
                os.path.basename(ben_file), 
                os.path.basename(adv_file), 
                snr, mcd, stoi, pesq
            ])
df = pd.DataFrame(res, columns=['lalm', 'attack', 'carrier_type', 'ben_audio_file', 'adv_audio_file', 'snr', 'mcd', 'stoi', 'pesq'])
df.to_csv(os.path.join(percept_dir, 'percept_result.csv'))

In [ ]:
df = pd.read_csv(os.path.join(percept_dir, 'percept_result.csv'))
l2_df = df[df['attack'] == 'caa_l2']
linf_df = df[df['attack'] == 'caa_linf']
cov_df = df[df['attack'] == 'caa']
print(linf_df.groupby(['carrier_type'])[['snr', 'mcd', 'stoi', 'pesq']].agg(['mean']))
print(l2_df.groupby(['carrier_type'])[['snr', 'mcd', 'stoi', 'pesq']].agg(['mean']))
print(cov_df.groupby(['carrier_type'])[['snr', 'mcd', 'stoi', 'pesq']].agg(['mean']))